# Dashboard Data Preparation

## Objective

This notebook prepares a consolidated dataset for the Power BI executive dashboard.

The dashboard focuses on four business areas:

- Sales performance
- Product category performance
- Customer behavior
- Delivery performance

The dataset is built from the cleaned relational tables exported during the data cleaning stage.

In [1]:
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed")

customers = pd.read_csv(
    processed_path / "customers.csv",
    dtype={
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "string"
    }
)

orders = pd.read_csv(
    processed_path / "orders.csv",
    dtype={
        "order_id": "string",
        "customer_id": "string"
    },
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv(
    processed_path / "order_items.csv",
    dtype={
        "order_id": "string",
        "product_id": "string",
        "seller_id": "string"
    }
)

products = pd.read_csv(
    processed_path / "products.csv",
    dtype={
        "product_id": "string"
    }
)

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)

customers: (99441, 8)
orders: (99441, 8)
order_items: (112650, 7)
products: (32951, 10)


## 1. Build the Dashboard Analysis Grain

The dashboard dataset uses one row per completed order and product category.

Order-item records are first aggregated to the order-category level to preserve category-level revenue analysis while reducing unnecessary item-level duplication.

Only delivered orders are included to maintain consistency with the sales, customer, and delivery analyses.

In [2]:
# Keep completed orders only
completed_orders = orders[
    orders["order_status"].eq("delivered")
].copy()

print("Completed orders:", len(completed_orders))

Completed orders: 96478


In [3]:
items_with_category = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name_english"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

items_with_category.shape

(112650, 8)

In [4]:
order_category = (
    items_with_category
    .groupby(
        [
            "order_id",
            "product_category_name_english"
        ],
        dropna=False,
        as_index=False
    )
    .agg(
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count")
    )
)

order_category.head()

,order_id,product_category_name_english,product_revenue,freight_value,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90,18.14,1


In [5]:
print("Order-category rows:", len(order_category))
print("Unique orders:", order_category["order_id"].nunique())

print(
    "Product revenue:",
    round(
        order_category[
            order_category["order_id"].isin(completed_orders["order_id"])
        ]["product_revenue"].sum(),
        2
    )
)

Order-category rows: 99470
Unique orders: 98666
Product revenue: 13221498.11


## 2. Add Order and Customer Attributes

Completed orders are joined with customer information to support time-based, customer, and delivery analysis.

Because the dashboard dataset is at the order-category level, order-level attributes may appear across multiple category rows. Order-level KPIs must therefore use distinct order or customer identifiers rather than raw row counts.

In [6]:
# Keep order-category records for completed orders only
dashboard_data = order_category.merge(
    completed_orders[
        [
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

print("Dashboard rows:", len(dashboard_data))
print("Completed orders:", dashboard_data["order_id"].nunique())
print(
    "Product revenue:",
    round(dashboard_data["product_revenue"].sum(), 2)
)

Dashboard rows: 97276
Completed orders: 96478
Product revenue: 13221498.11


In [7]:
dashboard_data = dashboard_data.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print(
    "Unique customers:",
    dashboard_data["customer_unique_id"].nunique()
)

dashboard_data.head()

Unique customers: 93358


,order_id,product_category_name_english,product_revenue,freight_value,item_count,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90,13.29,1,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,RJ
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90,19.93,1,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,SP
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00,17.87,1,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,MG
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99,12.79,1,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90,18.14,1,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,SP


## 3. Create Customer and Delivery Segments

Customer purchase frequency is calculated using `customer_unique_id`, which identifies the same customer across multiple orders.

Customers with one completed order are classified as `One-Time`, while customers with more than one completed order are classified as `Repeat`.

Delivery performance is evaluated using calendar dates rather than exact timestamps to avoid classifying orders delivered on the estimated date as late.

In [8]:
# Calculate completed-order frequency for each unique customer
customer_frequency = (
    completed_orders
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id"
            ]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    .groupby(
        "customer_unique_id",
        as_index=False
    )
    .agg(
        customer_completed_orders=("order_id", "nunique")
    )
)

customer_frequency["customer_type"] = (
    customer_frequency["customer_completed_orders"]
    .apply(
        lambda x: "One-Time" if x == 1 else "Repeat"
    )
)

customer_frequency["customer_type"].value_counts()

customer_type
One-Time    90557
Repeat       2801
Name: count, dtype: int64

In [9]:
dashboard_data = dashboard_data.merge(
    customer_frequency[
        [
            "customer_unique_id",
            "customer_type"
        ]
    ],
    on="customer_unique_id",
    how="left",
    validate="many_to_one"
)

dashboard_data["customer_type"].value_counts()

customer_type
One-Time    91256
Repeat       6020
Name: count, dtype: int64

In [10]:
delivered_date = (
    dashboard_data["order_delivered_customer_date"]
    .dt.normalize()
)

estimated_date = (
    dashboard_data["order_estimated_delivery_date"]
    .dt.normalize()
)

dashboard_data["delivery_status"] = pd.NA

dashboard_data.loc[
    delivered_date < estimated_date,
    "delivery_status"
] = "Early"

dashboard_data.loc[
    delivered_date == estimated_date,
    "delivery_status"
] = "On Estimated Date"

dashboard_data.loc[
    delivered_date > estimated_date,
    "delivery_status"
] = "Late"

In [11]:
delivery_check = (
    dashboard_data[
        [
            "order_id",
            "delivery_status"
        ]
    ]
    .drop_duplicates(subset="order_id")
)

delivery_check["delivery_status"].value_counts()

delivery_status
Early                88644
Late                  6534
On Estimated Date     1292
Name: count, dtype: int64

In [12]:
print(
    "Orders with delivery status:",
    delivery_check["delivery_status"].notna().sum()
)

Orders with delivery status: 96470


## 4. Create Dashboard Features

Additional fields are created for time-series analysis and delivery performance.

The dashboard uses purchase month for sales trends and calendar-date comparisons for delivery status.

In [13]:
# Purchase date and month
dashboard_data["purchase_date"] = (
    dashboard_data["order_purchase_timestamp"].dt.date
)

dashboard_data["purchase_month"] = (
    dashboard_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [14]:
dashboard_data["delivery_days"] = (
    (
        dashboard_data["order_delivered_customer_date"]
        - dashboard_data["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    / 86400
)

In [15]:
dashboard_data["days_early"] = (
    estimated_date - delivered_date
).dt.days.where(
    dashboard_data["delivery_status"].eq("Early")
)

dashboard_data["days_late"] = (
    delivered_date - estimated_date
).dt.days.where(
    dashboard_data["delivery_status"].eq("Late")
)

In [16]:
dashboard_data = dashboard_data[
    [
        "order_id",
        "customer_unique_id",
        "customer_state",
        "purchase_date",
        "purchase_month",
        "product_category_name_english",
        "product_revenue",
        "freight_value",
        "item_count",
        "customer_type",
        "delivery_status",
        "delivery_days",
        "days_early",
        "days_late"
    ]
].copy()

In [17]:
dashboard_data["product_category_name_english"] = (
    dashboard_data["product_category_name_english"]
    .fillna("Unknown")
)

In [18]:
print("=== Dashboard Dataset QA ===")

print("Rows:", len(dashboard_data))
print(
    "Completed orders:",
    dashboard_data["order_id"].nunique()
)
print(
    "Unique customers:",
    dashboard_data["customer_unique_id"].nunique()
)
print(
    "Product revenue:",
    round(dashboard_data["product_revenue"].sum(), 2)
)

order_level_check = (
    dashboard_data[
        [
            "order_id",
            "delivery_status",
            "delivery_days",
            "days_early",
            "days_late"
        ]
    ]
    .drop_duplicates(subset="order_id")
)

print(
    "Valid delivery orders:",
    order_level_check["delivery_status"].notna().sum()
)

print(
    "Average delivery days:",
    round(order_level_check["delivery_days"].mean(), 2)
)

print(
    "Average days early:",
    round(order_level_check["days_early"].mean(), 2)
)

print(
    "Average days late:",
    round(order_level_check["days_late"].mean(), 2)
)

=== Dashboard Dataset QA ===
Rows: 97276
Completed orders: 96478
Unique customers: 93358
Product revenue: 13221498.11
Valid delivery orders: 96470
Average delivery days: 12.56
Average days early: 13.71
Average days late: 10.62


In [19]:
dashboard_path = processed_path / "dashboard_data.csv"

dashboard_data.to_csv(
    dashboard_path,
    index=False
)

print("Dashboard dataset exported to:")
print(dashboard_path)

Dashboard dataset exported to:
../data/processed/dashboard_data.csv
